# РС2 — рекомендательная система фильмов

Реализованы задания: топ-10 популярных фильмов; фильтрация по фильмам и пользователям; контентные рекомендации; персональные рекомендации.

## 1. Импорт библиотек

In [ ]:
import re, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

## 2. Загрузка и обработка данных

In [ ]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings (1).csv")
ratings = ratings[["userId", "movieId", "rating"]].dropna()
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)
ratings["rating"] = ratings["rating"].astype(float)

def clean_title(title):
    title = re.sub(r"\\s*\\(\\d{4}\\)\\s*", " ", str(title))
    return re.sub(r"\\s+", " ", title).strip()
movies["title_original"] = movies["title"].astype(str)
movies["title_clean"] = movies["title_original"].map(clean_title)
movies["genres"] = movies["genres"].fillna("").replace("(no genres listed)", "")
stats = ratings.groupby("movieId").agg(rating_count=("rating","count"), mean_rating=("rating","mean")).reset_index()
movies = movies.merge(stats, on="movieId", how="left")
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)
movies["mean_rating"] = movies["mean_rating"].fillna(0.0)
print(f"Фильмов: {len(movies):,}; оценок: {len(ratings):,}; пользователей: {ratings.userId.nunique():,}")
display(movies[["movieId","title_clean","genres","rating_count","mean_rating"]].head())

Фильмов: 9,742; оценок: 100,836; пользователей: 610


,movieId,title_clean,genres,rating_count,mean_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215,3.920930
1,2,Jumanji (1995),Adventure|Children|Fantasy,110,3.431818
2,3,Grumpier Old Men (1995),Comedy|Romance,52,3.259615
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,7,2.357143
4,5,Father of the Bride Part II (1995),Comedy,49,3.071429


## 3. Поиск фильма без года

In [3]:
def find_movie(title):
    q = clean_title(title).casefold()
    if not q:
        raise ValueError("Введите название фильма.")
    found = movies[movies["title_clean"].str.casefold() == q]
    if found.empty:
        found = movies[movies["title_clean"].str.casefold().str.contains(re.escape(q), na=False)]
    if found.empty:
        raise ValueError(f"Фильм «{title}» не найден.")
    # Если совпадений несколько, выбираем более популярный вариант.
    return found.sort_values(["rating_count","mean_rating"], ascending=False).iloc[0]

for q in ["Toy Story", "Toy Story (1995)", "Fight Club"]:
    try:
        f = find_movie(q)
        print(f"{q!r} -> {f['title_clean']} (movieId={f['movieId']})")
    except ValueError as e:
        print(e)

'Toy Story' -> Toy Story (1995) (movieId=1)
'Toy Story (1995)' -> Toy Story (1995) (movieId=1)
'Fight Club' -> Fight Club (1999) (movieId=2959)


## 4. Задание 1 — топ-10 популярных фильмов

In [4]:
C = ratings["rating"].mean()
m = max(int(movies["rating_count"].quantile(.90)), 1)
movies["weighted_rating"] = (
    movies["rating_count"]/(movies["rating_count"]+m)*movies["mean_rating"]
    + m/(movies["rating_count"]+m)*C
)
def top_popular(n=10, min_votes=20):
    d = movies[movies.rating_count >= min_votes].sort_values(
        ["weighted_rating","rating_count"], ascending=False).head(n).copy()
    d["Название"] = d["title_clean"]
    return d[["movieId","Название","genres","rating_count","mean_rating","weighted_rating"]]
display(top_popular(10))

,movieId,Название,genres,rating_count,mean_rating,weighted_rating
277,318,"Shawshank Redemption, The (1994)",Crime|Drama,317,4.429022,4.356227
659,858,"Godfather, The (1972)",Crime|Drama,192,4.289062,4.191973
2226,2959,Fight Club (1999),Action|Crime|Drama|Thriller,218,4.272936,4.187927
224,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,251,4.231076,4.160223
46,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,204,4.237745,4.151697
461,527,Schindler's List (1993),Drama|War,220,4.225000,4.145919
257,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,307,4.197068,4.140844
898,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,211,4.215640,4.134630
1939,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,278,4.192446,4.131285
922,1221,"Godfather: Part II, The (1974)",Crime|Drama,129,4.259690,4.128475


## 5. Подготовка матрицы пользователь–фильм

In [5]:
user_item = ratings.pivot_table(index="userId", columns="movieId", values="rating")
movie_ids = user_item.columns.to_list()
movie_id_to_idx = {int(mid): i for i, mid in enumerate(movie_ids)}
print("Размер матрицы:", user_item.shape)

Размер матрицы: (610, 9724)


## 6. Задание 2 — коллаборативная фильтрация по фильмам

In [6]:
from scipy.sparse import csr_matrix
movie_similarity = cosine_similarity(csr_matrix(user_item.fillna(0).T.to_numpy()), dense_output=False)
def recommend_similar_movies(title, n=10):
    film = find_movie(title)
    mid = int(film.movieId)
    if mid not in movie_id_to_idx:
        raise ValueError("Для фильма недостаточно оценок.")
    sims = movie_similarity.getrow(movie_id_to_idx[mid]).toarray().ravel()
    order = np.argsort(sims)[::-1]
    order = [i for i in order if movie_ids[i] != mid][:n]
    ids = [movie_ids[i] for i in order]
    d = movies[movies.movieId.isin(ids)].copy()
    d["similarity"] = d.movieId.map({movie_ids[i]:float(sims[i]) for i in order})
    d["Название"] = d["title_clean"]
    return d.sort_values("similarity",ascending=False)[["movieId","Название","genres","similarity","rating_count"]].head(n)
display(recommend_similar_movies("Toy Story", 10))

,movieId,Название,genres,similarity,rating_count
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.572601,97
418,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.565637,238
615,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,0.564262,202
224,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.557388,251
314,356,Forrest Gump (1994),Comedy|Drama|Romance|War,0.547096,329
322,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.541145,172
911,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.541089,196
546,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,0.538913,162
964,1265,Groundhog Day (1993),Comedy|Fantasy|Romance,0.534169,143
969,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.530381,171


## 7. Задание 2 — коллаборативная фильтрация по пользователям

In [7]:
def recommend_for_user(user_id, n=10, neighbors=30):
    user_id = int(user_id)
    if user_id not in user_item.index:
        raise ValueError(f"Пользователь {user_id} не найден.")
    mat = user_item.fillna(0)
    sims = cosine_similarity(mat.loc[[user_id]], mat).ravel()
    nearest = pd.Series(sims, index=mat.index).drop(index=user_id).sort_values(ascending=False).head(neighbors)
    numerator = pd.Series(0.0, index=mat.columns)
    denominator = pd.Series(0.0, index=mat.columns)
    for other_id, sim in nearest.items():
        if sim <= 0: continue
        vals = user_item.loc[other_id]
        rated = vals.notna()
        numerator.loc[rated] += vals.loc[rated] * sim
        denominator.loc[rated] += sim
    scores = numerator / denominator.replace(0, np.nan)
    seen = user_item.loc[user_id].dropna().index
    scores = scores.drop(index=seen, errors="ignore").dropna().sort_values(ascending=False).head(n)
    d = movies[movies.movieId.isin(scores.index)].copy()
    d["Прогноз оценки"] = d.movieId.map(scores)
    d["Название"] = d["title_clean"]
    return d.sort_values("Прогноз оценки",ascending=False)[["movieId","Название","genres","Прогноз оценки"]].head(n)

example_user = int(ratings.userId.value_counts().index[0])
print("Пример пользователя:", example_user)
display(recommend_for_user(example_user, 10))

Пример пользователя: 414


,movieId,Название,genres,Прогноз оценки
4590,6818,Come and See (Idi i smotri) (1985),Drama|War,5.0
4782,7121,Adam's Rib (1949),Comedy|Romance,5.0
7491,82744,Faster (2010),Action|Crime|Drama,5.0
7656,88448,Paper Birds (Pájaros de papel) (2010),Comedy|Drama,5.0
7886,94810,Eva (2011),Drama|Fantasy|Sci-Fi,5.0
8551,115727,Crippled Avengers (Can que) (Return of the 5 D...,Action|Adventure,5.0
8839,132333,Seve (2014),Documentary|Drama,5.0
9147,147376,Doctor Who: A Christmas Carol (2010),Sci-Fi,5.0
9546,172875,A Detective Story (2003),Animation|Sci-Fi,5.0
9550,173145,War for the Planet of the Apes (2017),Action|Adventure|Drama|Sci-Fi,5.0


## 8. Задание 3 — контентные рекомендации по жанрам

In [8]:
tfidf = TfidfVectorizer(token_pattern=r"[^ ]+")
content_matrix = tfidf.fit_transform(movies["genres"].str.replace("|"," ",regex=False))
# Сходство вычисляем для одного фильма в функции, чтобы не создавать огромную квадратную матрицу.
def recommend_by_content(title, n=10):
    film = find_movie(title)
    idx = movies.index[movies.movieId == int(film.movieId)][0]
    sims = cosine_similarity(content_matrix[idx], content_matrix).ravel()
    order = np.argsort(sims)[::-1]
    order = [i for i in order if int(movies.iloc[i].movieId) != int(film.movieId)][:n]
    d = movies.iloc[order].copy()
    d["Сходство по жанрам"] = sims[order]
    d["Название"] = d["title_clean"]
    return d[["movieId","Название","genres","Сходство по жанрам","mean_rating"]]
display(recommend_by_content("Toy Story", 10))

,movieId,Название,genres,Сходство по жанрам,mean_rating
8219,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0,2.500000
3000,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.716216
2809,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0,2.222222
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.860825
6948,65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.000000
9430,166461,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.450000
7760,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0,5.000000
1706,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.244444
6486,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.023810
6194,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0,2.500000


## 9. Задание 4 — персональные рекомендации

In [9]:
def personalized_recommendations(user_id, n=10):
    # Основной источник — прогноз рейтинга на основе похожих пользователей.
    d = recommend_for_user(user_id, max(n*5, 50)).copy()
    if d.empty:
        return top_popular(n)
    d["popularity"] = d["movieId"].map(movies.set_index("movieId")["weighted_rating"]).fillna(C) / 5
    d["final_score"] = .85 * (d["Прогноз оценки"] / 5) + .15 * d["popularity"]
    return d.sort_values("final_score", ascending=False)[["movieId","Название","genres","Прогноз оценки","final_score"]].head(n)
display(personalized_recommendations(example_user, 10))

,movieId,Название,genres,Прогноз оценки,final_score
2852,3814,Love and Death (1975),Comedy,5.0,0.960452
4782,7121,Adam's Rib (1949),Comedy|Romance,5.0,0.959879
4590,6818,Come and See (Idi i smotri) (1985),Drama|War,5.0,0.958147
7364,78836,Enter the Void (2009),Drama,5.0,0.958147
6783,60333,Encounters at the End of the World (2008),Documentary,5.0,0.958042
3277,4437,Suspiria (1977),Horror,5.0,0.958036
9147,147376,Doctor Who: A Christmas Carol (2010),Sci-Fi,5.0,0.957630
6048,40412,Dead Man's Shoes (2004),Crime|Thriller,5.0,0.957630
4121,5915,Victory (a.k.a. Escape to Victory) (1981),Action|Drama|War,5.0,0.957630
2739,3677,Baraka (1992),Documentary,5.0,0.957460
